# Práctica: cuantizar un modelo con llama.cpp

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosqueteroweb/IAconceptos/blob/main/notebooks/cuantizacion.ipynb)

Vamos a tomar un modelo pequeño en su formato original (16 bits por parámetro) y a **comprimirlo** a 8, 4 y 2 bits
con [llama.cpp](https://github.com/ggml-org/llama.cpp), la misma herramienta que usan por dentro LM Studio y Ollama.
Después compararemos tamaño, velocidad y calidad de respuesta.


> **Cómo usar este cuaderno.** No hace falta saber programar. Ejecuta las celdas **en orden**, de arriba abajo,
> con el botón ▶ que aparece a su izquierda (o con **Mayús + Intro**). Lee el texto que hay antes de cada celda:
> explica qué va a pasar. Si algo falla, usa *Entorno de ejecución → Reiniciar y ejecutar todo*.


**No necesita GPU**: funciona en el entorno gratuito normal de Colab (CPU). Tarda unos 10 minutos.

El modelo es [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) (Alibaba, licencia Apache 2.0, sin registro).
Material del curso [IAconceptos](https://mosqueteroweb.github.io/IAconceptos/) · MosqueteroWeb · CC BY-SA 4.0.

## Paso 1. Instalar llama.cpp

Descargamos la última versión ya compilada de llama.cpp desde GitHub (no hay que compilar nada) y el *script* de
conversión, que está escrito en Python.

In [ ]:
import json, platform, subprocess, sys, tarfile, urllib.request, os
from pathlib import Path

sistema = platform.system()
sufijo = "bin-ubuntu-x64.tar.gz" if sistema == "Linux" else "bin-macos-arm64.tar.gz"
version = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=1"))[0]
paquete = next(a for a in version["assets"] if a["name"].endswith(sufijo) and a["name"].startswith("llama-"))
print("Descargando", paquete["name"])
urllib.request.urlretrieve(paquete["browser_download_url"], "llama.tar.gz")
with tarfile.open("llama.tar.gz") as t:
    t.extractall("llama", filter="data")
BIN = next(p.parent for p in Path("llama").rglob("llama-quantize"))
print("Programas en:", BIN)

# El script de conversión de la misma versión
etiqueta = version["tag_name"]
subprocess.run(["git", "clone", "--depth", "1", "--branch", etiqueta,
                "https://github.com/ggml-org/llama.cpp.git", "llama-src"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "llama-src/requirements/requirements-convert_hf_to_gguf.txt"], check=True)
print("Listo:", etiqueta)

## Paso 2. Descargar el modelo original

Bajamos el modelo tal y como lo publica su autor en Hugging Face: pesos en formato *safetensors* a 16 bits.

In [ ]:
from huggingface_hub import snapshot_download

carpeta = snapshot_download("Qwen/Qwen3-0.6B", local_dir="qwen-original")
tamano = sum(f.stat().st_size for f in Path(carpeta).glob("*.safetensors"))
print(f"Tamaño de los pesos originales: {tamano / 1e9:.2f} GB")

## Paso 3. Convertir a GGUF (sin cuantizar todavía)

**GGUF** es el formato de archivo único que entienden LM Studio, Ollama y llama.cpp: guarda en un solo fichero los
pesos, el tokenizador y la información de la arquitectura. Primero lo convertimos manteniendo los 16 bits (BF16).

In [ ]:
subprocess.run([sys.executable, "llama-src/convert_hf_to_gguf.py", "qwen-original",
                "--outtype", "bf16", "--outfile", "qwen-bf16.gguf"], check=True, capture_output=True)
print(f"qwen-bf16.gguf: {Path('qwen-bf16.gguf').stat().st_size / 1e9:.2f} GB")

## Paso 4. Cuantizar a varios niveles

Ahora sí: `llama-quantize` redondea cada peso a menos bits. Probamos tres niveles:

| Tipo | Bits aprox. | Qué es |
|---|---|---|
| `Q8_0` | 8,5 | Casi sin pérdida. Redondeo simple a 8 bits. |
| `Q4_K_M` | 4,8 | El equilibrio habitual: el que verás más en LM Studio. |
| `Q2_K` | 2,6 | Compresión extrema: se nota la pérdida de calidad. |

In [ ]:
tamanos = {"BF16": Path("qwen-bf16.gguf").stat().st_size / 1e6}
for tipo in ["Q8_0", "Q4_K_M", "Q2_K"]:
    salida = f"qwen-{tipo}.gguf"
    subprocess.run([str(BIN / "llama-quantize"), "qwen-bf16.gguf", salida, tipo], check=True, capture_output=True)
    tamanos[tipo] = Path(salida).stat().st_size / 1e6
    print(f"{tipo:7s} → {tamanos[tipo]:6.0f} MB  ({100 * tamanos[tipo] / tamanos['BF16']:.0f} % del original)")

## Paso 5. Comparar velocidad y respuestas

Hacemos la misma pregunta a cada versión y medimos cuántos **tokens por segundo** genera. Con temperatura 0 el modelo
elige siempre la palabra más probable, así las diferencias se deben solo a la cuantización.

In [ ]:
import re

PREGUNTA = "Explica en dos frases qué es la fotosíntesis."
# Plantilla de chat de Qwen3: así el modelo sabe que es una conversación (y /no_think le pide responder sin "pensar")
PROMPT = f"<|im_start|>user\n{PREGUNTA} /no_think<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
resultados = {}
for tipo in ["BF16", "Q8_0", "Q4_K_M", "Q2_K"]:
    archivo = "qwen-bf16.gguf" if tipo == "BF16" else f"qwen-{tipo}.gguf"
    r = subprocess.run([str(BIN / "llama-completion"), "-m", archivo, "-p", PROMPT, "-n", "120",
                        "--temp", "0", "--repeat-penalty", "1.1", "-no-cnv", "--no-warmup", "--no-display-prompt"],
                       capture_output=True, text=True)
    velocidad = re.search(r"eval time.*?([\d.]+) tokens per second", r.stderr.split("prompt eval time")[-1])
    resultados[tipo] = float(velocidad.group(1)) if velocidad else float("nan")
    print(f"=== {tipo} · {resultados[tipo]:.1f} tokens/s ===")
    print(r.stdout.strip()[:600], "\n")

In [ ]:
import matplotlib.pyplot as plt

tipos = list(resultados)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8))
a.bar(tipos, [tamanos[t] for t in tipos], color="#3b6fd8"); a.set_title("Tamaño del archivo (MB)")
b.bar(tipos, [resultados[t] for t in tipos], color="#14918a"); b.set_title("Velocidad (tokens/s)")
for eje in (a, b):
    eje.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Paso 6. Llévatelo a LM Studio

Descarga el archivo `qwen-Q4_K_M.gguf` (panel de archivos de la izquierda → tres puntos → Descargar). En LM Studio,
abre una terminal y ejecuta `lms import qwen-Q4_K_M.gguf`, o arrastra el archivo a la ventana de *My Models*.

**Para pensar:** ¿la versión `Q2_K` contesta igual de bien? ¿Merece la pena el ahorro? En modelos grandes (30 000
millones de parámetros o más) la pérdida de `Q4_K_M` apenas se nota; en uno tan pequeño como este, sí.